# Using SHARD

# Module import

In [1]:
# Standard libraries
import json
import os
from pathlib import Path

# Third-party libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from joblib import dump
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# PyTorch core
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader, TensorDataset

# PyTorch Lightning
import pytorch_lightning as pl
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

# TorchMetrics
from torchmetrics.classification import (
    BinaryAUROC,
    BinaryAveragePrecision,
    BinaryF1Score,
    BinaryPrecision,
    BinaryRecall
)

In [2]:
import glob, math, torch

# Data preparation

In [ ]:
class _PreProcessor:
    """Reusable object so we don't have to re-load norm.json for every shard."""
    def __init__(self, norm_json: str):
        with open(norm_json, "r") as f:
            self.norm = json.load(f)

    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (140, 11) tensor
        returns (140, 10) tensor
        """

        #original column [year, month, day, hour, minute, lat, lon, wp, tir, size, mask]

        # Drop the 'year' column → (140, 10)
        x = x[:, 1:]

        out = torch.empty((x.size(0), 10), dtype=torch.float32)

        # --- cyclic month (columns already shifted) ---------------------------
        month = x[:, 0]
        out[:, 0] = torch.sin(2 * math.pi * (month - 1) / 12.0)
        out[:, 1] = torch.cos(2 * math.pi * (month - 1) / 12.0)

        # 1 is just the day

        # --- cyclic time of day ----------------------------------------------
        hour, minute = x[:, 2], x[:, 3]
        tod = hour + minute / 60.0
        out[:, 2] = torch.sin(2 * math.pi * tod / 24.0)
        out[:, 3] = torch.cos(2 * math.pi * tod / 24.0)

        # --- lat/lon ----------------------------------------------------------
        lat, lon = x[:, 4], x[:, 5]
        out[:, 4] = (lat - self.norm["lat_min"]) / (self.norm["lat_max"] - self.norm["lat_min"])
        out[:, 5] = (lon - self.norm["lon_min"]) / (self.norm["lon_max"] - self.norm["lon_min"])

        # --- wavelet power (log-scaled) --------------------------------------
        wp = x[:, 6]
        out[:, 6] = torch.log1p(wp) / self.norm["wp_max"]

        # --- TIR --------------------------------------------------------------
        tir = x[:, 7]
        out[:, 7] = (tir - self.norm["tir_min"]) / (self.norm["tir_max"] - self.norm["tir_min"])

        # --- size -------------------------------------------------------------
        size = x[:, 8]
        out[:, 8] = torch.log1p(size) / self.norm["size_max"]

        # --- mask (unchanged) -------------------------------------------------
        out[:, 9] = x[:, 9]

        return out

In [ ]:
@torch.no_grad()
def build_single_file(
    shards_dir: str,
    norm_path: str,
    out_path: str = "nowcasting_train.pt",
    glob_pattern: str = "*.pt",
):
    """
    Iterate over every shard (each holding (x,y)), apply preprocessing once,
    then save **one** file with keys:
        • inputs  → (N, 140, 10)
        • targets → (N, 1024, 1024)
    """
    processor = _PreProcessor(norm_path)
    shard_files = sorted(glob.glob(os.path.join(shards_dir, glob_pattern)))
    if not shard_files:
        raise FileNotFoundError("No shard files found!")

    xs, ys = [], []
    for fp in tqdm(shard_files, desc="Merging shards"):
        data = torch.load(fp, map_location="cpu")
        x, y = data["inputs"], data["targets"]
        xs.append(processor(x.float()))
        ys.append(y.float())

    inputs  = torch.stack(xs)           # (N, 140, 10)
    targets = torch.stack(ys)           # (N, 1024, 1024)
    torch.save({"inputs": inputs, "targets": targets}, out_path)
    print(f"✅  Wrote {inputs.size(0):,} samples to {out_path}")

In [9]:
root_dir = "/gws/nopw/j04/wiser_ewsa/mrakotomanga/EPS/Africa_sharded/t1/training"
norm_path = "/home/users/mendrika/EPS-Impact-Case-AI-Nowcasting/model/africa/normalisation/parameters/normalisation.json"

In [ ]:
build_single_file(root_dir, norm_path)

Merging shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [5]:
# ------------------------------------------------------------------
# 3.  ──  Dataset that reads the merged file  ──
# ------------------------------------------------------------------
class StormNowcastingDataset(Dataset):
    """
    Parameters
    ----------
    data_path : path to the *.pt produced by `build_single_file`
    mmap      : if True and data was saved with torch.save(..., _use_new_zipfile_serialization=True),
                tensors are opened lazily, keeping RAM low.
    transform : optional callable(inputs, targets) → tuple
    """

    def __init__(self, data_path: str, mmap: bool = False, transform=None):
        load_fn = torch.load if not mmap else lambda p: torch.load(p, mmap=True)
        blob = load_fn(data_path, map_location="cpu")

        self.inputs  = blob["inputs"]   # (N, 140, 10)
        self.targets = blob["targets"]  # (N, 1024, 1024)
        self.transform = transform

    def __len__(self):
        return self.inputs.size(0)

    def __getitem__(self, idx):
        x, y = self.inputs[idx], self.targets[idx]
        if self.transform is not None:
            x, y = self.transform(x, y)
        return x, y

In [ ]:
root_dir = "/gws/nopw/j04/wiser_ewsa/mrakotomanga/EPS/Africa_sharded/t1/training"
norm_path = "/home/users/mendrika/EPS-Impact-Case-AI-Nowcasting/model/africa/normalisation/parameters/normalisation.json"

train_file_list = "/home/users/mendrika/EPS-Impact-Case-AI-Nowcasting/model/africa/splits/train_files.csv"

train_dataset = StormNowcastingDataset(
    root_dir=root_dir,
    norm_path=norm_path,
    file_list=train_file_list,
    lead_time=1
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=4
)

for batch_inputs, batch_targets in train_loader:
    print(batch_inputs.shape)  # (B, 140, 10)
    print(batch_targets.shape) # (B, 1024, 1024)
    break